## Task 1.2 — Technical Analysis & Trading System
### Installing and Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
import os
warnings.filterwarnings('ignore')
os.makedirs('charts_task2', exist_ok=True)
print("Libraries imported successfully")

### Downloading Stock Data — Using HDFCBANK, TCS, SUNPHARMA

In [ ]:
tickers = {
    'BANK_HDFC':  'HDFCBANK.NS',
    'IT_TCS':     'TCS.NS',
    'PHARMA_SUN': 'SUNPHARMA.NS',
}
start_date = '2019-01-01'
end_date   = '2024-12-31'

raw_data = {}
for label, ticker in tickers.items():
    df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    # Flatten multi-level columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.ffill()
    raw_data[label] = df
    print(f"{label}: shape={df.shape}, {df.index.min().date()} to {df.index.max().date()}")

## Part A — Technical Indicator Library
### Indicator 1: Simple Moving Average (SMA) — Lagging Indicator

In [ ]:
def compute_sma(df, windows=[20, 50]):
    """
    Simple Moving Average (SMA)
    Formula: SMA(n) = rolling mean of Adjusted Close over n periods
    Measures: Average price over n days — smooths out noise
    Type: LAGGING — based entirely on past prices
    Best for: Identifying trend direction, support/resistance levels
    """
    for w in windows:
        df[f'SMA_{w}'] = df['Adj Close'].rolling(w).mean()
    return df

def plot_sma(df, label):
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(df.index, df['Adj Close'], label='Adj Close', color='steelblue', linewidth=1)
    ax.plot(df.index, df['SMA_20'], label='SMA-20', color='orange', linewidth=1.5, linestyle='--')
    ax.plot(df.index, df['SMA_50'], label='SMA-50', color='red', linewidth=1.5, linestyle='--')

    # Mark crossover points
    cross_up = (df['SMA_20'] > df['SMA_50']) & (df['SMA_20'].shift(1) <= df['SMA_50'].shift(1))
    cross_dn = (df['SMA_20'] < df['SMA_50']) & (df['SMA_20'].shift(1) >= df['SMA_50'].shift(1))
    ax.scatter(df.index[cross_up], df['Adj Close'][cross_up], marker='^', color='green', s=100, zorder=5, label='Bullish Cross')
    ax.scatter(df.index[cross_dn], df['Adj Close'][cross_dn], marker='v', color='red', s=100, zorder=5, label='Bearish Cross')

    ax.set_title(f'{label} — SMA-20 and SMA-50 with Crossovers', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (INR)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_SMA.png', dpi=150)
    plt.show()
    print(f"Saved: charts_task2/{label}_SMA.png")

# Run for all stocks
for label, df in raw_data.items():
    raw_data[label] = compute_sma(df)
    plot_sma(raw_data[label], label)

### Indicator 2: Exponential Moving Average (EMA) — Lagging Indicator

In [ ]:
def compute_ema(df, windows=[12, 26]):
    """
    Exponential Moving Average (EMA)
    Formula: EMA(n) = ewm(span=n, adjust=False).mean()
    Measures: Weighted average giving more weight to recent prices
    Type: LAGGING — but reacts faster than SMA to recent price changes
    Best for: Trend following, MACD calculation, faster signals than SMA
    """
    for w in windows:
        df[f'EMA_{w}'] = df['Adj Close'].ewm(span=w, adjust=False).mean()
    return df

def plot_ema(df, label):
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(df.index, df['Adj Close'], label='Adj Close', color='steelblue', linewidth=1)
    ax.plot(df.index, df['EMA_12'], label='EMA-12', color='green', linewidth=1.5)
    ax.plot(df.index, df['EMA_26'], label='EMA-26', color='red', linewidth=1.5)
    ax.plot(df.index, df['SMA_20'], label='SMA-20', color='orange', linewidth=1.5, linestyle='--', alpha=0.7)
    ax.set_title(f'{label} — EMA-12 vs EMA-26 vs SMA-20 (Reaction Speed Comparison)', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (INR)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_EMA.png', dpi=150)
    plt.show()

for label, df in raw_data.items():
    raw_data[label] = compute_ema(df)
    plot_ema(raw_data[label], label)

### Indicator 3: Relative Strength Index (RSI) — Lagging/Leading Hybrid

In [ ]:
def compute_rsi(df, n=14):
    """
    Relative Strength Index (RSI) — Wilder Smoothing
    Formula: RSI = 100 - (100 / (1 + RS)) where RS = Avg Gain / Avg Loss
    Measures: Speed and magnitude of price movements — momentum oscillator
    Type: LAGGING (based on past prices) but used as LEADING for reversals
    Best for: Identifying overbought (>70) and oversold (<30) conditions
    Wilder smoothing uses alpha = 1/n instead of standard EWM
    """
    delta = df['Adj Close'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/n, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/n, adjust=False).mean()
    rs  = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

def plot_rsi(df, label):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    ax1.plot(df.index, df['Adj Close'], color='steelblue', label='Adj Close')
    ax1.set_title(f'{label} — Price and RSI', fontweight='bold')
    ax1.set_ylabel('Price (INR)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(df.index, df['RSI'], color='purple', linewidth=1.2, label='RSI-14')
    ax2.axhline(70, color='red', linestyle='--', linewidth=1, label='Overbought (70)')
    ax2.axhline(30, color='green', linestyle='--', linewidth=1, label='Oversold (30)')
    ax2.fill_between(df.index, 70, 100, alpha=0.1, color='red')
    ax2.fill_between(df.index, 0, 30, alpha=0.1, color='green')
    ax2.set_ylabel('RSI')
    ax2.set_xlabel('Date')
    ax2.set_ylim(0, 100)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_RSI.png', dpi=150)
    plt.show()

for label, df in raw_data.items():
    raw_data[label] = compute_rsi(df)
    plot_rsi(raw_data[label], label)

### Indicator 4: MACD — Lagging Indicator

In [ ]:
def compute_macd(df):
    """
    Moving Average Convergence Divergence (MACD)
    MACD Line   = EMA(12) - EMA(26)
    Signal Line = EMA(9) of MACD Line
    Histogram   = MACD Line - Signal Line
    Measures: Momentum and trend direction changes
    Type: LAGGING — based on EMAs which are themselves lagging
    Best for: Trend confirmation and crossover-based entry/exit signals
    Bullish crossover: MACD crosses above Signal Line
    Bearish crossover: MACD crosses below Signal Line
    """
    df['MACD_Line']   = df['EMA_12'] - df['EMA_26']
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist']   = df['MACD_Line'] - df['Signal_Line']
    return df

def plot_macd(df, label):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    ax1.plot(df.index, df['Adj Close'], color='steelblue', label='Adj Close')
    ax1.set_title(f'{label} — Price and MACD', fontweight='bold')
    ax1.set_ylabel('Price (INR)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(df.index, df['MACD_Line'], label='MACD Line', color='blue', linewidth=1.2)
    ax2.plot(df.index, df['Signal_Line'], label='Signal Line', color='red', linewidth=1.2)

    # Histogram — green positive, red negative
    colors = ['green' if v >= 0 else 'red' for v in df['MACD_Hist']]
    ax2.bar(df.index, df['MACD_Hist'], color=colors, width=1, alpha=0.5, label='Histogram')

    # Mark crossovers
    bull = (df['MACD_Line'] > df['Signal_Line']) & (df['MACD_Line'].shift(1) <= df['Signal_Line'].shift(1))
    bear = (df['MACD_Line'] < df['Signal_Line']) & (df['MACD_Line'].shift(1) >= df['Signal_Line'].shift(1))
    ax2.scatter(df.index[bull], df['MACD_Line'][bull], marker='^', color='green', s=80, zorder=5, label='Bullish Cross')
    ax2.scatter(df.index[bear], df['MACD_Line'][bear], marker='v', color='red', s=80, zorder=5, label='Bearish Cross')

    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_ylabel('MACD')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_MACD.png', dpi=150)
    plt.show()

for label, df in raw_data.items():
    raw_data[label] = compute_macd(df)
    plot_macd(raw_data[label], label)

### Indicator 5: Bollinger Bands — Lagging Indicator

In [ ]:
def compute_bollinger(df, n=20):
    """
    Bollinger Bands
    Middle Band = SMA(20)
    Upper Band  = SMA(20) + 2 * rolling std(20)
    Lower Band  = SMA(20) - 2 * rolling std(20)
    Measures: Price volatility and relative high/low levels
    Type: LAGGING — based on SMA and historical standard deviation
    Best for: Identifying volatility regimes and mean reversion signals
    Band Squeeze: When bands narrow (low std), it signals low volatility
    and predicts an upcoming large move (breakout) in either direction
    """
    df['BB_Mid']   = df['Adj Close'].rolling(n).mean()
    df['BB_Std']   = df['Adj Close'].rolling(n).std()
    df['BB_Upper'] = df['BB_Mid'] + 2 * df['BB_Std']
    df['BB_Lower'] = df['BB_Mid'] - 2 * df['BB_Std']
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Mid']
    return df

def plot_bollinger(df, label):
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.plot(df.index, df['Adj Close'], label='Adj Close', color='steelblue', linewidth=1)
    ax.plot(df.index, df['BB_Mid'],   label='Middle Band (SMA-20)', color='orange', linewidth=1.2)
    ax.plot(df.index, df['BB_Upper'], label='Upper Band', color='red', linewidth=1, linestyle='--')
    ax.plot(df.index, df['BB_Lower'], label='Lower Band', color='green', linewidth=1, linestyle='--')
    ax.fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.1, color='gray', label='Band Region')
    ax.set_title(f'{label} — Bollinger Bands (20, 2)', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (INR)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_Bollinger.png', dpi=150)
    plt.show()

for label, df in raw_data.items():
    raw_data[label] = compute_bollinger(df)
    plot_bollinger(raw_data[label], label)

### Combined Multi-Panel Chart — BANK_HDFC (Price + RSI + MACD + Volume)

In [ ]:
def plot_combined(df, label):
    fig, axes = plt.subplots(4, 1, figsize=(18, 20), sharex=True)
    fig.suptitle(f'{label} — Combined Technical Analysis Dashboard', fontsize=16, fontweight='bold')

    # Panel 1: Price + SMAs + Bollinger Bands
    axes[0].plot(df.index, df['Adj Close'], color='steelblue', linewidth=1, label='Price')
    axes[0].plot(df.index, df['SMA_20'], color='orange', linewidth=1.2, linestyle='--', label='SMA-20')
    axes[0].plot(df.index, df['SMA_50'], color='red', linewidth=1.2, linestyle='--', label='SMA-50')
    axes[0].plot(df.index, df['BB_Upper'], color='gray', linewidth=0.8, linestyle=':')
    axes[0].plot(df.index, df['BB_Lower'], color='gray', linewidth=0.8, linestyle=':')
    axes[0].fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.1, color='gray')
    axes[0].set_ylabel('Price (INR)')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Panel 2: RSI
    axes[1].plot(df.index, df['RSI'], color='purple', linewidth=1.2)
    axes[1].axhline(70, color='red', linestyle='--', linewidth=1)
    axes[1].axhline(30, color='green', linestyle='--', linewidth=1)
    axes[1].fill_between(df.index, 70, 100, alpha=0.1, color='red')
    axes[1].fill_between(df.index, 0, 30, alpha=0.1, color='green')
    axes[1].set_ylabel('RSI')
    axes[1].set_ylim(0, 100)
    axes[1].grid(True, alpha=0.3)

    # Panel 3: MACD
    axes[2].plot(df.index, df['MACD_Line'], color='blue', linewidth=1.2, label='MACD')
    axes[2].plot(df.index, df['Signal_Line'], color='red', linewidth=1.2, label='Signal')
    colors = ['green' if v >= 0 else 'red' for v in df['MACD_Hist']]
    axes[2].bar(df.index, df['MACD_Hist'], color=colors, width=1, alpha=0.5)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].set_ylabel('MACD')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    # Panel 4: Volume
    vol = df['Volume'] if not isinstance(df['Volume'], pd.DataFrame) else df['Volume'].iloc[:, 0]
    axes[3].bar(df.index, vol, color='steelblue', width=1, alpha=0.6)
    axes[3].set_ylabel('Volume')
    axes[3].set_xlabel('Date')
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_combined.png', dpi=150)
    plt.show()
    print(f"Saved: charts_task2/{label}_combined.png")

plot_combined(raw_data['BANK_HDFC'], 'BANK_HDFC')

## Part B — Backtesting Engine
### EMA Crossover Signal Generation (No Look-Ahead Bias)

In [ ]:
def generate_signals(df, short=20, long=50):
    """
    EMA Crossover Signal Generation
    Buy  (+1): EMA-short crosses ABOVE EMA-long
    Sell (-1): EMA-short crosses BELOW EMA-long
    CRITICAL: Signal on day t is executed on day t+1 open
    This prevents look-ahead bias
    """
    df = df.copy()
    df[f'EMA_{short}'] = df['Adj Close'].ewm(span=short, adjust=False).mean()
    df[f'EMA_{long}']  = df['Adj Close'].ewm(span=long,  adjust=False).mean()

    df['Signal'] = 0
    cross_up = (df[f'EMA_{short}'] > df[f'EMA_{long}']) & (df[f'EMA_{short}'].shift(1) <= df[f'EMA_{long}'].shift(1))
    cross_dn = (df[f'EMA_{short}'] < df[f'EMA_{long}']) & (df[f'EMA_{short}'].shift(1) >= df[f'EMA_{long}'].shift(1))
    df.loc[cross_up, 'Signal'] = 1
    df.loc[cross_dn, 'Signal'] = -1

    # SHIFT signal by 1 day — execute next day to avoid look-ahead bias
    df['Signal_Exec'] = df['Signal'].shift(1).fillna(0)
    return df

for label in raw_data:
    raw_data[label] = generate_signals(raw_data[label])
    buys  = (raw_data[label]['Signal'] == 1).sum()
    sells = (raw_data[label]['Signal'] == -1).sum()
    print(f"{label}: {buys} buy signals, {sells} sell signals")
    

### Trade Simulation with Transaction Costs

In [ ]:
def backtest(df, label, initial_capital=100000, cost_pct=0.001):
    """
    Backtesting Engine
    - Executes signals at NEXT DAY open price (no look-ahead bias)
    - Invests ALL cash on Buy, liquidates ALL shares on Sell
    - Deducts 0.1% transaction cost on every trade leg
    - Tracks: cash, shares, portfolio value, trade log
    """
    df = df.copy().reset_index()
    cash      = initial_capital
    shares    = 0
    trade_log = []
    portfolio = []

    for i in range(len(df)):
        signal = df['Signal_Exec'].iloc[i]
        price  = df['Open'].iloc[i] if 'Open' in df.columns else df['Adj Close'].iloc[i]
        date   = df['Date'].iloc[i] if 'Date' in df.columns else df.index[i]

        if signal == 1 and cash > 0:
            cost       = cash * cost_pct
            cash      -= cost
            shares     = cash / price
            cash       = 0
            port_val   = shares * price
            trade_log.append({'Date': date, 'Action': 'BUY', 'Price': round(price,2),
                               'Shares': round(shares,4), 'Cost': round(cost,2),
                               'Portfolio Value': round(port_val,2)})

        elif signal == -1 and shares > 0:
            proceeds   = shares * price
            cost       = proceeds * cost_pct
            cash       = proceeds - cost
            port_val   = cash
            trade_log.append({'Date': date, 'Action': 'SELL', 'Price': round(price,2),
                               'Shares': round(shares,4), 'Cost': round(cost,2),
                               'Portfolio Value': round(port_val,2)})
            shares = 0

        port_val = cash + shares * price
        portfolio.append(port_val)

    df['Portfolio'] = portfolio
    trade_df = pd.DataFrame(trade_log)
    print(f"\n{label} — Trade Log:")
    print(trade_df.to_string(index=False))
    return df, trade_df

results = {}
trade_logs = {}
for label in raw_data:
    results[label], trade_logs[label] = backtest(raw_data[label], label)

### Performance Metrics — CAGR, Sharpe, Max Drawdown, Win Rate

In [ ]:
def compute_metrics(df, trade_df, label, initial_capital=100000, rf=0.06):
    """
    Performance Metrics:
    CAGR       = (Final/Initial)^(1/years) - 1
    Sharpe     = (mean daily return - rf/252) / std daily return * sqrt(252)
    Max DD     = largest peak-to-trough % decline in equity curve
    Win Rate   = profitable round trips / total round trips
    """
    port = df['Portfolio']
    years = (df.index[-1] - df.index[0]).days / 365.25 if isinstance(df.index[-1], pd.Timestamp) else len(df) / 252

    # CAGR
    cagr = (port.iloc[-1] / initial_capital) ** (1 / years) - 1

    # Sharpe
    daily_ret = port.pct_change().dropna()
    sharpe    = (daily_ret.mean() - rf/252) / daily_ret.std() * np.sqrt(252)

    # Max Drawdown
    roll_max = port.cummax()
    drawdown = (port - roll_max) / roll_max
    max_dd   = drawdown.min()

    # Win Rate
    wins = 0
    total = 0
    buys  = trade_df[trade_df['Action'] == 'BUY'].reset_index(drop=True)
    sells = trade_df[trade_df['Action'] == 'SELL'].reset_index(drop=True)
    for i in range(min(len(buys), len(sells))):
        total += 1
        if sells['Price'].iloc[i] > buys['Price'].iloc[i]:
            wins += 1
    win_rate = wins / total if total > 0 else 0

    # Buy and Hold
    bh_final = initial_capital * (df['Adj Close'].iloc[-1] / df['Adj Close'].iloc[0])
    bh_cagr  = (bh_final / initial_capital) ** (1 / years) - 1
    bh_ret   = df['Adj Close'].pct_change().dropna()
    bh_sharpe = (bh_ret.mean() - rf/252) / bh_ret.std() * np.sqrt(252)
    bh_roll   = (initial_capital * (1 + bh_ret).cumprod()).cummax()
    bh_dd     = ((initial_capital * (1 + bh_ret).cumprod()) - bh_roll) / bh_roll
    bh_maxdd  = bh_dd.min()

    metrics = pd.DataFrame({
        'Metric':         ['Final Value', 'CAGR', 'Sharpe Ratio', 'Max Drawdown', 'Win Rate'],
        'EMA Strategy':   [f"₹{port.iloc[-1]:,.0f}", f"{cagr:.2%}", f"{sharpe:.2f}", f"{max_dd:.2%}", f"{win_rate:.2%}"],
        'Buy and Hold':   [f"₹{bh_final:,.0f}",      f"{bh_cagr:.2%}", f"{bh_sharpe:.2f}", f"{bh_maxdd:.2%}", "N/A"]
    })
    print(f"\n{label} — Performance Comparison:")
    print(metrics.to_string(index=False))
    return metrics, drawdown

metrics_all = {}
drawdown_all = {}
for label in results:
    metrics_all[label], drawdown_all[label] = compute_metrics(results[label], trade_logs[label], label)

### Visualisation — Equity Curve, Drawdown Chart, Trade Markers

In [ ]:
def plot_backtest(df, trade_df, drawdown, label, initial_capital=100000):
    fig, axes = plt.subplots(3, 1, figsize=(16, 18))
    fig.suptitle(f'{label} — Backtest Results', fontsize=16, fontweight='bold')

    # Panel 1: Equity curve vs Buy and Hold (indexed to 100)
    port_idx = df['Portfolio'] / initial_capital * 100
    bh_idx   = (df['Adj Close'] / df['Adj Close'].iloc[0]) * 100
    axes[0].plot(df.index, port_idx, label='EMA Strategy', color='steelblue', linewidth=1.5)
    axes[0].plot(df.index, bh_idx,   label='Buy and Hold',  color='black', linewidth=1.5, linestyle='--')
    axes[0].axhline(100, color='gray', linestyle=':', linewidth=0.8)
    axes[0].set_title('Equity Curve vs Buy and Hold (Indexed to 100)')
    axes[0].set_ylabel('Portfolio Value (Indexed)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Panel 2: Drawdown chart
    axes[1].fill_between(df.index, drawdown * 100, 0, color='red', alpha=0.4)
    axes[1].plot(df.index, drawdown * 100, color='red', linewidth=1)
    axes[1].set_title('Drawdown Chart (% decline from peak)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].grid(True, alpha=0.3)

    # Panel 3: Price with buy/sell markers
    axes[2].plot(df.index, df['Adj Close'], color='steelblue', linewidth=1, label='Price')
    buys  = trade_df[trade_df['Action'] == 'BUY']
    sells = trade_df[trade_df['Action'] == 'SELL']
    axes[2].scatter(pd.to_datetime(buys['Date']),  buys['Price'],  marker='^', color='green', s=100, zorder=5, label='Buy')
    axes[2].scatter(pd.to_datetime(sells['Date']), sells['Price'], marker='v', color='red',   s=100, zorder=5, label='Sell')
    axes[2].set_title('Price Chart with Trade Signals')
    axes[2].set_ylabel('Price (INR)')
    axes[2].set_xlabel('Date')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'charts_task2/{label}_backtest.png', dpi=150)
    plt.show()
    print(f"Saved: charts_task2/{label}_backtest.png")

for label in results:
    plot_backtest(results[label], trade_logs[label], drawdown_all[label], label)

### Bonus — EMA Sensitivity Test: Window Combinations

In [ ]:
ema_combos = [(10, 30), (20, 50), (15, 45)]
sensitivity = []

for short, long in ema_combos:
    for label in raw_data:
        df_sig = generate_signals(raw_data[label].copy(), short=short, long=long)
        df_bt, tdf = backtest(df_sig, label)
        years = len(df_bt) / 252
        port  = df_bt['Portfolio']
        cagr  = (port.iloc[-1] / 100000) ** (1/years) - 1
        dr    = port.pct_change().dropna()
        sharpe = (dr.mean() - 0.06/252) / dr.std() * np.sqrt(252)
        roll_max = port.cummax()
        max_dd = ((port - roll_max) / roll_max).min()
        sensitivity.append({
            'Stock': label, 'Short EMA': short, 'Long EMA': long,
            'CAGR': f"{cagr:.2%}", 'Sharpe': f"{sharpe:.2f}", 'Max DD': f"{max_dd:.2%}"
        })

sens_df = pd.DataFrame(sensitivity)
print("\nEMA Sensitivity Table:")
print(sens_df.to_string(index=False))

## Part C — Strategy Comparison Report
### Strategy Comparison Table

In [ ]:
strategies = pd.DataFrame([
    {
        'Strategy':        'Momentum (EMA Crossover)',
        'Time Horizon':    '1–6 months',
        'Win Rate Range':  '40–55%',
        'Typical Sharpe':  '0.5–1.2',
        'Best Condition':  'Strong trending market',
        'Worst Condition': 'Choppy/sideways market',
        'Automation':      'High — rule-based'
    },
    {
        'Strategy':        'Mean Reversion (BB/RSI)',
        'Time Horizon':    '1–10 days',
        'Win Rate Range':  '55–70%',
        'Typical Sharpe':  '0.8–1.5',
        'Best Condition':  'Range-bound market',
        'Worst Condition': 'Trending market',
        'Automation':      'High — clear thresholds'
    },
    {
        'Strategy':        'Breakout',
        'Time Horizon':    '1–4 weeks',
        'Win Rate Range':  '35–50%',
        'Typical Sharpe':  '0.6–1.0',
        'Best Condition':  'High volume breakouts',
        'Worst Condition': 'Low volume, false breaks',
        'Automation':      'Medium — needs volume filter'
    },
    {
        'Strategy':        'Swing Trading',
        'Time Horizon':    '2–10 days',
        'Win Rate Range':  '45–60%',
        'Typical Sharpe':  '0.7–1.3',
        'Best Condition':  'Volatile trending market',
        'Worst Condition': 'Low volatility flat market',
        'Automation':      'Medium — needs discretion'
    },
    {
        'Strategy':        'Trend Following',
        'Time Horizon':    '3–12 months',
        'Win Rate Range':  '35–45%',
        'Typical Sharpe':  '0.5–1.0',
        'Best Condition':  'Long sustained trends',
        'Worst Condition': 'Whipsaw/choppy market',
        'Automation':      'High — systematic'
    },
    {
        'Strategy':        'Scalping',
        'Time Horizon':    'Seconds–minutes',
        'Win Rate Range':  '60–80%',
        'Typical Sharpe':  '1.5–3.0',
        'Best Condition':  'High liquidity, tight spreads',
        'Worst Condition': 'Wide spreads, low volume',
        'Automation':      'Very High — requires algos'
    }
])

print("Strategy Comparison Table:")
print(strategies.to_string(index=False))

### Part C — Written Analysis


**Feasibility Ranking for Retail Traders (most to least feasible):**
1. Mean Reversion — clear rules, moderate frequency, low capital needed
2. Swing Trading — manageable trade frequency, works on EOD data
3. Momentum/EMA Crossover — simple, automatable, tested in Part B
4. Trend Following — low trade frequency but requires patience
5. Breakout — needs real-time volume data, harder to automate
6. Scalping — requires co-location, near-zero costs, NOT feasible for retail

**Regime Dependency — When to Switch Off a Strategy:**
- EMA Crossover: Switch off when ADX < 20 (market not trending)
- Mean Reversion: Switch off when ADX > 30 (market strongly trending)
- Breakout: Switch off during low-volume periods (e.g. Indian market holidays)
- Monitor: 20-day rolling volatility, VIX India index, and ADX weekly